<a href="https://colab.research.google.com/github/niikun/ezkl/blob/main/simple_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EZKLを使用したゼロ知識証明（ZKP）によるFashion MNIST分類

このノートブックでは、**EZKL**ライブラリを使用して、機械学習モデルの推論をゼロ知識証明（Zero-Knowledge Proofs）で検証する一連の流れを解説します。

### ゼロ知識証明（ZKP）とは？
ある情報（この場合はAIの推論結果）が正しいことを、その根拠となる詳細なデータや計算プロセスを完全に明かすことなく数学的に証明する技術です。

### このワークフローの目的
1.  **AIモデルの信頼性**: 特定の学習済みモデルが、改ざんされることなく正しく実行されたことを証明します。
2.  **プライバシー保護**: 入力データ（画像など）を秘密にしたまま、その判定結果だけを正しい証拠と共に提示することが可能になります。

### ステップの概要
1.  **モデルの構築と訓練**: PyTorchでFashion MNIST（服の画像データセット）用のCNNモデルを作成し学習させます。
2.  **ONNXエクスポート**: 学習済みモデルをEZKLが扱える共通形式（ONNX）に変換します。
3.  **EZKL設定とキャリブレーション**: 回路の設計図を作成し、整数計算のための最適化を行います。
4.  **セットアップと鍵生成**: 証明に必要な数学的基盤（SRS）と、証明用・検証用の鍵（PK/VK）を作成します。
5.  **証明の生成と検証**: 実際の推論に対して「証明書」を発行し、それが数学的に正しいことを検証します。

In [4]:
# Google Colab環境かどうかの確認
try:
    # ezklのインストール
    import google.colab
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ezkl"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "onnx"])

# Colabでない場合は、ローカルにインストールされたezklを使用
except:
    pass

# モデルの作成（および訓練）の準備

# 必要な依存関係がインストールされていることを確認
from torch import nn
import ezkl
import os
import json
import torch

In [5]:
if torch.cuda.is_available():          
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f"device: {device}")

device: cpu


In [51]:
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=1, out_channels=2, kernel_size=5, stride=2)
        self.conv2 = nn.Conv2d(in_channels=2, out_channels=3, kernel_size=5, stride=2)

        self.relu = nn.ReLU()

        self.d1 = nn.Linear(48, 48)
        self.d2 = nn.Linear(48, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.relu(x)

        # ONNXとの互換性を高めるため、明示的なリシェイプを使用
        x = x.view(-1, 48)

        x = self.d1(x)
        x = self.relu(x)

        logits = self.d2(x)

        return logits

# 訓練とエクスポート用にモデルのインスタンスを作成
model = MyModel()

In [52]:
import torchvision
import torch

# 画像の変換設定
transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.5,), (0.5,)) # 1チャンネル画像用に[-1, 1]の範囲に正規化
])

# Fashion MNISTデータセットのロード
fashion_mnist_dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)

# DataLoaderの作成
data_loader = torch.utils.data.DataLoader(fashion_mnist_dataset,
                                          batch_size=32,
                                          shuffle=True,
                                          num_workers=2) # デフォルトのワーカー数を使用

In [60]:
model = MyModel()
# 損失関数（交差エントロピー）と最適化手法（Adam）の定義
loss = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### 学習の改善
現在の設定では精度が出ていないため、最適化アルゴリズムの設定を見直し、確実に勾配が更新されるようにします。

In [63]:
import torch.optim as optim

# モデルの再初期化（必要に応じて）
model = MyModel()

# 損失関数と最適化手法を定義
# 学習率(lr)を調整し、Adamなどの頑健なオプティマイザを使用します
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("モデルとオプティマイザを再設定しました。")

モデルとオプティマイザを再設定しました。


In [64]:
# 訓練ループ
num_epochs = 3  # エポック数は調整可能です

for epoch in range(num_epochs):
    running_loss = 0.0
    for i, data in enumerate(data_loader, 0):
        inputs, labels = data

        # パラメータの勾配をゼロにリセット
        optimizer.zero_grad()

        # 順伝播 + 逆伝播 + 最適化
        outputs = model(inputs)
        loss_value = loss(outputs, labels)
        loss_value.backward()
        optimizer.step()

        running_loss += loss_value.item()
        if i % 100 == 99:    # 100ミニバッチごとに表示
            print(f'Epoch {epoch + 1}, Batch {i + 1:5d} loss: {running_loss / 100:.3f}')
            running_loss = 0.0

print('訓練完了')

Epoch 1, Batch   100 loss: 1.898
Epoch 1, Batch   200 loss: 1.106
Epoch 1, Batch   300 loss: 0.968
Epoch 1, Batch   400 loss: 0.829
Epoch 1, Batch   500 loss: 0.771
Epoch 1, Batch   600 loss: 0.735
Epoch 1, Batch   700 loss: 0.703
Epoch 1, Batch   800 loss: 0.716
Epoch 1, Batch   900 loss: 0.697
Epoch 1, Batch  1000 loss: 0.689
Epoch 1, Batch  1100 loss: 0.663
Epoch 1, Batch  1200 loss: 0.647
Epoch 1, Batch  1300 loss: 0.648
Epoch 1, Batch  1400 loss: 0.644
Epoch 1, Batch  1500 loss: 0.634
Epoch 1, Batch  1600 loss: 0.623
Epoch 1, Batch  1700 loss: 0.615
Epoch 1, Batch  1800 loss: 0.595
Epoch 2, Batch   100 loss: 0.598
Epoch 2, Batch   200 loss: 0.620
Epoch 2, Batch   300 loss: 0.584
Epoch 2, Batch   400 loss: 0.588
Epoch 2, Batch   500 loss: 0.580
Epoch 2, Batch   600 loss: 0.583
Epoch 2, Batch   700 loss: 0.564
Epoch 2, Batch   800 loss: 0.563
Epoch 2, Batch   900 loss: 0.583
Epoch 2, Batch  1000 loss: 0.544
Epoch 2, Batch  1100 loss: 0.535
Epoch 2, Batch  1200 loss: 0.541
Epoch 2, B

After training, you might want to save the trained model weights or evaluate its performance on a test set. Let's do a quick evaluation on a few batches from the data loader.

In [65]:
# モデルの評価（数バッチ分のみの例）
correct = 0
total = 0
# 訓練ではないため、出力に対する勾配計算は不要
with torch.no_grad():
    for data in data_loader:
        images, labels = data
        # ネットワークに画像を入力して出力を計算
        outputs = model(images)
        # 最も高い値を持つクラスを予測結果とする
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'{total} 枚の画像に対するネットワークの精度: {100 * correct // total} %')

60000 枚の画像に対するネットワークの精度: 81 %


## ファイルパスの設定
EZKLの各ステップで生成されるモデル、設定、鍵、証明書などのファイルパスを定義します。

In [66]:
model_path = os.path.join('network.onnx')
compiled_model_path = os.path.join('network.compiled')
pk_path = os.path.join('test.pk')
vk_path = os.path.join('test.vk')
settings_path = os.path.join('settings.json')
witness_path = os.path.join('witness.json')
data_path = os.path.join('input.json')

## 1. ONNXエクスポート
訓練したPyTorchモデルを、EZKLが読み込める共通形式であるONNXに変換します。あわせて、推論に使用する入力データ（`input.json`）も準備します。

In [67]:
import torch
import json

# 前のステップで訓練した'model'インスタンスを使用
# 評価モードに設定
model.eval()

# エクスポート用のダミー入力を準備
shape = [1, 1, 28, 28]
x = torch.randn(1, *shape[1:], requires_grad=True)

# 互換性を確保するため、レガシー（TorchScriptベース）エクスポーターを使用してエクスポート
torch.onnx.export(model,
                  x,
                  model_path,
                  export_params=True,
                  opset_version=10,
                  do_constant_folding=True,
                  input_names=['input'],
                  output_names=['output'],
                  dynamo=False)

# ezkl用のinput.jsonを作成
data_array = ((x).detach().numpy()).reshape([-1]).tolist()
data = dict(input_data = [data_array])
json.dump(data, open(data_path, 'w'))

/tmp/ipykernel_1413/1373349736.py:13: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model,


### 🔍 ONNXモデルの中身をのぞいてみる
PyTorchで書いたコードが、どのような「計算グラフ」に変換されたかを確認してみます。ONNXは、モデルを「入力→畳み込み→ReLU→...→出力」という一連のノード（計算ステップ）として表現します。

In [68]:
import onnx

# 保存したONNXファイルを読み込みます
onnx_model = onnx.load(model_path)

# モデルの構造を簡潔に表示します
print(f"ONNX IR バージョン: {onnx_model.ir_version}")
print(f"プロデューサー: {onnx_model.producer_name} {onnx_model.producer_version}")

# 最初の数個の計算ノード（操作）を表示してみます
print("\n--- 最初の5つの計算ステップ ---")
for i, node in enumerate(onnx_model.graph.node[:5]):
    print(f"ステップ {i+1}: 操作={node.op_type}, 入力={node.input}, 出力={node.output}")

ONNX IR バージョン: 5
プロデューサー: pytorch 2.11.0

--- 最初の5つの計算ステップ ---
ステップ 1: 操作=Conv, 入力=['input', 'conv1.weight', 'conv1.bias'], 出力=['/conv1/Conv_output_0']
ステップ 2: 操作=Relu, 入力=['/conv1/Conv_output_0'], 出力=['/relu/Relu_output_0']
ステップ 3: 操作=Conv, 入力=['/relu/Relu_output_0', 'conv2.weight', 'conv2.bias'], 出力=['/conv2/Conv_output_0']
ステップ 4: 操作=Relu, 入力=['/conv2/Conv_output_0'], 出力=['/relu_1/Relu_output_0']
ステップ 5: 操作=Constant, 入力=[], 出力=['/Constant_output_0']


## 2. 設定ファイルの生成
モデルの構造を解析し、ゼロ知識証明の回路として動かすためのパラメータ（計算の階層やデータの公開範囲など）を定義した`settings.json`を作成します。

In [56]:
import ezkl

py_run_args = ezkl.PyRunArgs()
py_run_args.input_visibility = "public"
py_run_args.output_visibility = "public"
py_run_args.param_visibility = "fixed"

# 簡略化されたONNXグラフで設定生成を再試行
try:
    res = ezkl.gen_settings(model_path, settings_path, py_run_args=py_run_args)
    print(f"設定生成結果: {res}")
except Exception as e:
    print(f"生成失敗: {e}")

設定生成結果: True


### 💡 コラム：なぜ「回路」と呼ぶのか？

AIモデル（ONNX）をコンパイルすると、内部では以下のような変換が行われています。

1.  **計算の分解**: `y = Wx + b` というニューラルネットワークの計算を、最小単位の「足し算」と「掛け算」にバラバラにします。
2.  **制約への変換**: 「この入力とこの重みを掛けたら、必ずこの値にならなければならない」という数学的なルール（制約）のリストを作ります。
3.  **証明の生成**: 証明者は、入力データを隠したまま、すべてのルール（制約）が矛盾なく満たされていることだけを証明します。

つまり、**「計算を実行すること」を「数学的なパズルを解くこと」に置き換えている**のです。このパズルの構造が電子回路の設計図に似ているため、「回路（Circuit）」と呼ばれます。

### 🚀 モデルの規模と計算コスト（スケーラビリティ）

「大きなモデルでも証明できるのか？」という疑問に対する現在の状況は以下の通りです。

- **現在の限界**: 巨大なResNetやTransformer（LLMなど）をそのまま一つの回路にすると、証明の生成に数百GBのメモリが必要になったり、時間がかかりすぎたりします。
- **EZKLの対策**:
    - **量子化**: データの精度を適切に落とすことで、計算（回路サイズ）を劇的に軽量化します。
    - **集約 (Aggregation)**: 複数の小さな証明を一つにまとめ、検証コストを下げます。
    - **専用ハードウェア**: GPUやFPGAを使用して、証明生成を高速化する研究が進んでいます。

現実的には、エッジデバイスで動くような小型のCNNや、特定のタスクに特化したモデルが、現在のZKP（ゼロ知識証明）の主な活用対象となっています。

### 🧩 大規模モデルへの現実的なアプローチ：部分証明

Transformerのような巨大なモデルで、計算の全プロセスを証明するのが難しい場合、以下のような「ハイブリッド構成」が検討されます。

1.  **入力の同一性証明**:
    - ユーザーが送ったデータが、モデルの「最初の層」に正しく入力されたことを証明します（ハッシュ値の照合など）。
2.  **中間層のブラックボックス化**:
    - 巨大な中間層（Transformerの多層ブロックなど）は、通常のGPUサーバーで高速に計算します（ここは証明しません）。
3.  **出力の正当性証明**:
    - 最後の層（分類ヘッドなど）だけをZKP回路にし、「ある中間計算結果を受け取って、この最終判定を出した」というプロセスだけを証明します。

**この方法のメリット:**
- 回路サイズを数千分の一に削減でき、実用的な時間で証明が終わります。

**この方法の課題:**
- 「中間層で不正が行われていないか（モデルがすり替えられていないか）」をどう保証するかが新たな論点になります。これには「モデルの重みに対するコミットメント」という技術を組み合わせて対処します。

## 3. キャリブレーション（量子化調整）
浮動小数点のモデルを、証明回路で扱える形式（整数計算）に変換するためのスケール調整を行います。これにより、計算精度と証明の複雑さのバランスを最適化します。

In [57]:
cal_path = os.path.join("calibration.json")

data_array = (torch.rand(20, *shape, requires_grad=True).detach().numpy()).reshape([-1]).tolist()

data = dict(input_data = [data_array])

# データをファイルにシリアライズ:
json.dump(data, open(cal_path, 'w'))

# 量子化スケールのキャリブレーションを実行
ezkl.calibrate_settings(cal_path, model_path, settings_path, "resources")

ERROR:ezkl.graph.model:[tensor] decomposition error: integer -1769582246 is too large to be represented by base 16384 and n 2
ERROR:ezkl.execute:forward pass failed: "failed to forward: [halo2] General synthesis error"
ERROR:ezkl.graph.model:[tensor] decomposition error: integer -7053700227 is too large to be represented by base 16384 and n 2
ERROR:ezkl.execute:forward pass failed: "failed to forward: [halo2] General synthesis error"
ERROR:ezkl.graph.model:[tensor] decomposition error: integer -28206125210 is too large to be represented by base 16384 and n 2
ERROR:ezkl.execute:forward pass failed: "failed to forward: [halo2] General synthesis error"
ERROR:ezkl.graph.model:[tensor] decomposition error: integer -14106829407 is too large to be represented by base 16384 and n 2
ERROR:ezkl.execute:forward pass failed: "failed to forward: [halo2] General synthesis error"
ERROR:ezkl.graph.model:[tensor] decomposition error: integer -56409937531 is too large to be represented by base 16384 and

True

## 4. コンパイル
ONNXモデルと設定ファイルを組み合わせて、EZKL専用の実行形式（回路）にコンパイルします。

In [58]:
import ezkl

# 1. 回路をコンパイルします
res = ezkl.compile_circuit(model_path, compiled_model_path, settings_path)
assert res == True
print("回路のコンパイルが成功しました。")

回路のコンパイルが成功しました。


## 5. 証拠（Witness）の生成
実際の入力データを使用してモデルを「実行」し、証明の構築に必要となる計算過程のすべての値を記録した証拠ファイルを作成します。

In [59]:
# 2. 証拠（Witness）を生成します
# 先ほど作成したinput.jsonを使用します
res = ezkl.gen_witness(data_path, compiled_model_path, witness_path)
assert os.path.exists(witness_path)
print("証拠の生成が成功しました。")

証拠の生成が成功しました。


## 6. セットアップ
数学的な土台となるSRSを準備し、証明に必要な「証明鍵(PK)」と「検証鍵(VK)」を生成します。検証鍵は、後に誰でも内容を検証できるように公開されるものです。

In [61]:
# 3. セットアップ (SRSのダウンロードと鍵生成)
import ezkl
import os

# settings.jsonの"logrows"に基づいた、回路サイズに応じたSRSファイルをダウンロードします
ezkl.get_srs(settings_path)

# 証明鍵 (PK) と検証鍵 (VK) を生成します
# 注意: これには数分かかる場合があります
res = ezkl.setup(
    compiled_model_path,
    vk_path,
    pk_path,
)

assert res == True
assert os.path.exists(vk_path)
assert os.path.exists(pk_path)
print("セットアップ完了: VKおよびPKが生成されました。")

セットアップ完了: VKおよびPKが生成されました。


## 7. 証明の生成と検証
最後に、証拠と証明鍵を使って「正しく計算したこと」の証明書（`proof.json`）を発行し、それが正しいことを検証鍵でチェックします。

In [62]:
proof_path = os.path.join('proof.json')

# 4. 証明の生成 (Prove)
# 証拠ファイル、コンパイル済みモデル、証明鍵を使用して、推論の正当性を示す証明書を作成します
res = ezkl.prove(
    witness=witness_path,
    model=compiled_model_path,
    pk_path=pk_path,
    proof_path=proof_path,
)

# 証明ファイルが生成されていることを確認します
assert os.path.exists(proof_path)
print("証明書の生成が完了しました。")

# 5. 検証 (Verify)
# 検証鍵を使用して、生成された証明書が数学的に正しいかチェックします
res_verify = ezkl.verify(
    proof_path=proof_path,
    settings_path=settings_path,
    vk_path=vk_path
)

assert res_verify == True
print("検証成功: この推論結果は数学的に正しいことが証明されました！")

証明書の生成が完了しました。
検証成功: この推論結果は数学的に正しいことが証明されました！
